# Module 1: Pothole and crack detector (RDD2022) — Colab training

Run top to bottom. **Runtime → Change runtime type → T4 GPU** first.

What this does:
1. Mounts your Google Drive (checkpoints and exports are saved there, so a disconnect doesn't lose work)
2. Clones the project repo and installs dependencies
3. Downloads RDD2022 from Kaggle (needs your `kaggle.json` token)
4. Converts it to YOLO format with our 4 classes (`detector/configs/classes.yaml`)
5. Trains a small YOLO model, resuming automatically if a previous run exists
6. Evaluates on all countries and on India only
7. Exports TFLite (FP16 and INT8) and saves a results summary to Drive

Dataset: RDD2022, Arya et al., CC BY-SA 4.0 — attribution required in the report.


## 0. Settings — edit these if needed

In [ ]:
MODEL = "yolo11n.pt"      # or "yolov8n.pt"; check the name against the current Ultralytics docs
IMGSZ = 640
EPOCHS = 50
BATCH = 16                # lower to 8 if you hit GPU out-of-memory
FRACTION = 1.0            # set to 0.25 for a quick first baseline run
RUN_NAME = "rdd_pothole_v1"
KAGGLE_DATASET = "aliabdelmenam/rdd-2022"
DRIVE_DIR = "/content/drive/MyDrive/road-anomaly"
RAW_DIR = "/content/rdd_raw"
YOLO_DIR = "/content/rdd_yolo"
REPO = "https://github.com/divyanshkulshrestha23/road-anomaly-detection"


## 1. GPU check and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Saving outputs to", DRIVE_DIR)


## 2. Clone repo and install

In [ ]:
import os
if not os.path.exists("/content/road-anomaly-detection"):
    !git clone -q {REPO} /content/road-anomaly-detection
else:
    !git -C /content/road-anomaly-detection pull -q
%cd /content/road-anomaly-detection
!pip install -q -r detector/requirements.txt
!pip install -q -U kaggle
!git log --oneline -1


## 3. Kaggle token

On kaggle.com: **Settings → API → Generate New Token**, then copy the token string.

Recommended: in Colab, click the **key icon (Secrets)** in the left sidebar, **Add new secret**, name it `KAGGLE_API_TOKEN`, paste the token as the value, and switch on **Notebook access**.
If you skip that, this cell asks you to paste the token in a hidden box.
It's a credential: don't commit it or paste it into chat.

In [ ]:
import os, getpass
token = None
try:
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    pass
if not token and not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    token = getpass.getpass("Paste your Kaggle API token: ").strip()
if token:
    os.environ["KAGGLE_API_TOKEN"] = token
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    tpath = os.path.expanduser("~/.kaggle/access_token")
    with open(tpath, "w") as f:
        f.write(token)
    os.chmod(tpath, 0o600)
!kaggle --version
!kaggle datasets files -d {KAGGLE_DATASET} | head -5


## 4. Download RDD2022 (~11 GB, to Colab's local disk, not Drive)

In [ ]:
import os
if not os.path.exists(RAW_DIR) or not os.listdir(RAW_DIR):
    os.makedirs(RAW_DIR, exist_ok=True)
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content/rdd_zip
    !unzip -q -o /content/rdd_zip/*.zip -d {RAW_DIR}
    !rm -rf /content/rdd_zip   # free disk space
else:
    print("Already downloaded")
!df -h /content | tail -1


## 5. Inspect the dataset layout
Check the output: countries should include India, and class names should look like D00/D10/D20/D40 or their spelled-out names.

In [ ]:
!python detector/rdd_prepare.py --src {RAW_DIR} --inspect

## 6. Convert to YOLO format
Keeps pothole, longitudinal, transverse and alligator cracks. Drops other classes. Excludes the China drone (aerial) subset.

In [ ]:
!python detector/rdd_prepare.py --src {RAW_DIR} --out {YOLO_DIR}
!cp {YOLO_DIR}/prepare_summary.json {YOLO_DIR}/split_manifest.csv {DRIVE_DIR}/


## 7. Train (resumes automatically after a disconnect)

In [ ]:
import os
from ultralytics import YOLO
project = f"{DRIVE_DIR}/runs"
last = f"{project}/{RUN_NAME}/weights/last.pt"
if os.path.exists(last):
    print("Resuming from", last)
    model = YOLO(last)
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(data=f"{YOLO_DIR}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                fraction=FRACTION, project=project, name=RUN_NAME, exist_ok=True,
                patience=15, seed=42, device=0)


## 8. Evaluate: all countries, then India only

In [ ]:
from ultralytics import YOLO
best = f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt"
model = YOLO(best)
results = {}
for tag, data in [("all", f"{YOLO_DIR}/data.yaml"), ("india", f"{YOLO_DIR}/data_india.yaml")]:
    for split in ["val", "test"]:
        m = model.val(data=data, split=split, imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
        per_class = {model.names[i]: round(float(m.box.maps[i]), 4) for i in range(len(model.names))}
        results[f"{tag}_{split}"] = {"mAP50": round(float(m.box.map50), 4),
                                     "mAP50_95": round(float(m.box.map), 4),
                                     "precision": round(float(m.box.mp), 4),
                                     "recall": round(float(m.box.mr), 4),
                                     "per_class_mAP50_95": per_class}
        print(tag, split, results[f"{tag}_{split}"])


## 9. Export to TFLite (FP16 and INT8) and copy to Drive

In [ ]:
import glob, os, shutil
from ultralytics import YOLO
export_dir = f"{DRIVE_DIR}/exports/{RUN_NAME}"
os.makedirs(export_dir, exist_ok=True)
for kw in [dict(half=True), dict(int8=True, data=f"{YOLO_DIR}/data.yaml")]:
    try:
        YOLO(best).export(format="tflite", imgsz=IMGSZ, **kw)
    except Exception as e:
        print("Export failed for", kw, "->", e)
for f in glob.glob(f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/**/*.tflite", recursive=True):
    shutil.copy2(f, export_dir)
print(os.listdir(export_dir))


## 10. Save summary to Drive
Download `summary.json` and send it to Claude, or commit it to `eval/results/`.

In [ ]:
import json, datetime, subprocess, torch
summary = {
    "run": RUN_NAME, "model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "batch": BATCH,
    "fraction": FRACTION, "date": datetime.datetime.now().isoformat(timespec="seconds"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip(),
    "dataset": "RDD2022 (Kaggle: " + KAGGLE_DATASET + "), CC BY-SA 4.0",
    "targets": {"pothole_mAP50": 0.5},
    "results": results,
    "exports": sorted(os.listdir(export_dir)),
}
with open(f"{DRIVE_DIR}/summary_{RUN_NAME}.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
